# LeetCode 3845 — Maximum Subarray XOR with Bounded Range bằng Python

- Link: https://leetcode.com/problems/maximum-subarray-xor-with-bounded-range/
- Độ khó: **Hard**
- Chủ đề: XOR tiền tố, cửa sổ trượt với hàng đợi đơn điệu (max/min), trie nhị phân có đếm (thêm/xóa), trie bền vững (persistent trie)

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def maxXor(self, nums: list[int], k: int) -> int:
```

## 1. Mục tiêu học

1. Viết XOR của đoạn `nums[l..r]` thành `P[r+1] ⊕ P[l]` với `P` là **XOR tiền tố**.
2. Duy trì cửa sổ thỏa `max − min ≤ k` bằng **hai deque đơn điệu**; hiểu vì sao đầu trái hợp lệ là một **khoảng liên tiếp** `[L(r), r]` và `L(r)` không giảm.
3. Tìm `max (x ⊕ y)` với `y` trong một tập động bằng **trie nhị phân** có bộ đếm (hỗ trợ thêm và xóa).
4. Thay thế bằng **trie bền vững** trả lời "max XOR với tiền tố có chỉ số trong `[L, r]`".
5. Kiểm thử với vét cạn `O(n²)`.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Cho mảng không âm `nums` và số `k`. Chọn một **đoạn con** (liên tiếp, khác rỗng) sao cho `max − min ≤ k`. Giá trị của đoạn là XOR mọi phần tử. Tìm giá trị lớn nhất.

**Ví dụ 1:** `nums = [5,4,5,6], k = 2` → `7` (cả mảng: `max − min = 2`, `5⊕4⊕5⊕6 = 7`).

**Ví dụ 2:** `nums = [5,4,5,6], k = 1` → `6` (đoạn `[6]`).

**Ràng buộc:** `1 ≤ n ≤ 4·10^4`, `0 ≤ nums[i] < 2^{15}`, `0 ≤ k < 2^{15}`.

In [1]:
from collections import deque
import random, time

examples = [([5, 4, 5, 6], 2, 7), ([5, 4, 5, 6], 1, 6)]
for nums, k, exp in examples:
    print(nums, "k =", k, "-> kỳ vọng", exp)

[5, 4, 5, 6] k = 2 -> kỳ vọng 7
[5, 4, 5, 6] k = 1 -> kỳ vọng 6


## 3. Kiến thức nền

### 3.1. XOR tiền tố

`P[0] = 0`, `P[i+1] = P[i] ⊕ nums[i]`. Vì `a ⊕ a = 0`: `nums[l] ⊕ … ⊕ nums[r] = P[r+1] ⊕ P[l]`.

In [2]:
nums = [5, 4, 5, 6]
P = [0]
for x in nums:
    P.append(P[-1] ^ x)
print("P =", P, "| XOR nums[1..3] =", 4 ^ 5 ^ 6, "=", P[4] ^ P[1])

P = [0, 5, 1, 4, 2] | XOR nums[1..3] = 7 = 7


### 3.2. Deque đơn điệu cho max/min trên cửa sổ trượt

- `maxq` giữ chỉ số với giá trị **giảm dần**: phần tử mới đẩy bỏ các chỉ số có giá trị `≤` nó; đầu deque là max.
- `minq` tương tự với giá trị **tăng dần**.
- Khi đầu trái cửa sổ vượt qua chỉ số ở đầu deque thì `popleft`.

In [3]:
def window_max(a, w):
    dq, out = deque(), []
    for i, x in enumerate(a):
        while dq and a[dq[-1]] <= x:
            dq.pop()
        dq.append(i)
        if dq[0] <= i - w:
            dq.popleft()
        if i >= w - 1:
            out.append(a[dq[0]])
    return out

print(window_max([1, 3, -1, -3, 5, 3, 6, 7], 3))

[3, 3, 5, 5, 6, 7]


### 3.3. Trie nhị phân tìm max XOR

Lưu các số (15 bit) theo bit từ cao xuống thấp. Để tối đa `x ⊕ y`, tại mỗi bit cố đi sang nhánh **ngược** với bit của `x`. Thêm bộ đếm `cnt` ở mỗi nút để hỗ trợ **xóa** (giảm đếm) — nhánh có `cnt = 0` coi như không tồn tại.

In [4]:
B = 15
class Trie:
    def __init__(self):
        self.ch = [[0, 0]]      # nút 0 là gốc; ch[u][b] = chỉ số nút con (0 = chưa có)
        self.cnt = [0]
    def update(self, v, d):
        u = 0
        for bit in range(B - 1, -1, -1):
            b = v >> bit & 1
            if not self.ch[u][b]:
                self.ch.append([0, 0]); self.cnt.append(0)
                self.ch[u][b] = len(self.ch) - 1
            u = self.ch[u][b]
            self.cnt[u] += d
    def best(self, v):
        u, res = 0, 0
        for bit in range(B - 1, -1, -1):
            b = v >> bit & 1
            o = self.ch[u][b ^ 1]
            if o and self.cnt[o] > 0:
                res |= 1 << bit; u = o
            else:
                u = self.ch[u][b]
        return res

t = Trie()
for v in (3, 10, 5, 25 % 32):
    t.update(v, 1)
print("max XOR với 2:", t.best(2), "| kiểm tra:", max(2 ^ v for v in (3, 10, 5, 25)))
t.update(10, -1)
print("sau khi xóa 10:", t.best(2), "| kiểm tra:", max(2 ^ v for v in (3, 5, 25)))

max XOR với 2: 27 | kiểm tra: 27
sau khi xóa 10: 27 | kiểm tra: 27


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Cố định đầu phải `r`. Đoạn `[l, r]` hợp lệ khi `max − min ≤ k`. Khi `l` giảm (đoạn dài ra), `max` chỉ tăng và `min` chỉ giảm ⇒ hiệu chỉ tăng ⇒ các `l` hợp lệ tạo thành **khoảng** `[L(r), r]`. Khi `r` tăng, `L(r)` không giảm ⇒ cửa sổ trượt.

Giá trị đoạn = `P[r+1] ⊕ P[l]` với `l ∈ [L(r), r]`. Vậy với mỗi `r`: **tìm `max (P[r+1] ⊕ y)` với `y ∈ {P[L], …, P[r]}`** — một truy vấn max-XOR trên tập động.

Tính tay ví dụ 2 (`[5,4,5,6]`, `k = 1`), `P = [0,5,1,4,2]`:

| r | cửa sổ hợp lệ | tập `P[l]` | `P[r+1]` | max XOR |
|---|---|---|---|---|
| 0 | [0,0] | {P0=0} | 5 | 5⊕0 = 5 |
| 1 | [0,1] (5,4: hiệu 1) | {0, 5} | 1 | max(1⊕0, 1⊕5) = 4 |
| 2 | [0,2] (5,4,5) | {0, 5, 1} | 4 | max(4, 1, 5) = 5 |
| 3 | [2,3] (5,6) — phải bỏ 5 và 4 ở đầu vì 6 − 4 = 2 > 1 | {P2=1, P3=4} | 2 | max(3, 6) = **6** |

Đáp án `6` ✔ (đoạn `[6]`: `P4 ⊕ P3 = 2 ⊕ 4 = 6`).

## 5. Cách 1 — Brute Force `O(n²)`

Cố định `l`, mở rộng `r`, cập nhật XOR, max, min; dừng khi `max − min > k` (đoạn dài hơn cũng hỏng).

In [5]:
def brute(nums, k):
    n = len(nums); best = 0
    for l in range(n):
        x = 0; mx = -1; mn = 1 << 30
        for r in range(l, n):
            v = nums[r]
            x ^= v; mx = max(mx, v); mn = min(mn, v)
            if mx - mn > k:
                break
            best = max(best, x)
    return best

for nums, k, exp in examples:
    print(brute(nums, k), "kỳ vọng", exp)

7 kỳ vọng 7
6 kỳ vọng 6


**Nút thắt:** `O(n²) = 1,6·10^9` khi `k` lớn (mọi đoạn hợp lệ). Cần trả lời "max XOR trong cửa sổ" nhanh.

## 6. Ý tưởng thuật toán / bước đột phá

**Kết hợp hai kỹ thuật:**

1. **Cửa sổ trượt + deque đơn điệu** cho `L(r)`: thêm `r` vào `maxq`, `minq`; trong khi `nums[maxq[0]] − nums[minq[0]] > k` thì tăng `L` (bỏ chỉ số `L` khỏi đầu deque nếu trùng).
2. **Trie có đếm** chứa đúng tập `{P[L], …, P[r]}`: khi tăng `L` thì **xóa** `P[L]`; trước khi xử lý `r+1` thì **thêm** `P[r+1]`.

Mỗi `P[i]` vào và ra trie một lần ⇒ `O(n · 15)`.

**Quy tắc thứ tự:** trước khi truy vấn cho đầu phải `r`, trie phải chứa `P[L..r]` (gồm `P[r]` nhưng **chưa** có `P[r+1]` — vì đoạn khác rỗng).

## 7. Các cách tiếp theo

- **Cách 2:** cửa sổ trượt cho `L(r)` + quét tuyến tính `max_{l ∈ [L, r]} P[r+1] ⊕ P[l]` — `O(n · w)`.
- **Cách 3:** cửa sổ trượt + **trie có đếm** — `O(n · B)` (lời giải nộp chính).
- **Cách 4:** tính `L(r)` trước, rồi dùng **trie bền vững** (mỗi phiên bản lưu chỉ số lớn nhất trong cây con) để truy vấn trên khoảng chỉ số — `O(n · B)`.

### 7.1. Cách 2 — Cửa sổ trượt + quét tuyến tính

In [6]:
def left_bounds(nums, k):
    """L[r] = đầu trái nhỏ nhất sao cho max - min của nums[L..r] <= k."""
    maxq, minq = deque(), deque()
    L = 0; res = []
    for r, x in enumerate(nums):
        while maxq and nums[maxq[-1]] <= x: maxq.pop()
        while minq and nums[minq[-1]] >= x: minq.pop()
        maxq.append(r); minq.append(r)
        while nums[maxq[0]] - nums[minq[0]] > k:
            if maxq[0] == L: maxq.popleft()
            if minq[0] == L: minq.popleft()
            L += 1
        res.append(L)
    return res

def window_scan(nums, k):
    P = [0]
    for x in nums: P.append(P[-1] ^ x)
    Ls = left_bounds(nums, k)
    return max(max(P[r + 1] ^ P[l] for l in range(Ls[r], r + 1)) for r in range(len(nums)))

print("L(r) ví dụ 2:", left_bounds([5, 4, 5, 6], 1))
for nums, k, exp in examples:
    print(window_scan(nums, k), "kỳ vọng", exp)

L(r) ví dụ 2: [0, 0, 0, 2]
7 kỳ vọng 7
6 kỳ vọng 6


**Độ phức tạp:** `O(n · w)` với `w` là độ rộng cửa sổ — tệ nhất vẫn `O(n²)`.

### 7.2. Cách 3 — Cửa sổ trượt + trie có đếm (lời giải chính)

**Giải thích từng đoạn:**

1. `P` tiền tố; `trie.update(P[0], +1)`.
2. Với mỗi `r`: cập nhật deque; trong khi vi phạm thì `trie.update(P[L], −1)` và `L += 1`.
3. `ans = max(ans, trie.best(P[r+1]))`, rồi `trie.update(P[r+1], +1)`.

In [7]:
def trie_window(nums, k):
    n = len(nums)
    P = [0] * (n + 1)
    for i, x in enumerate(nums): P[i + 1] = P[i] ^ x
    t = Trie(); t.update(P[0], 1)
    maxq, minq = deque(), deque()
    L = 0; ans = 0
    for r, x in enumerate(nums):
        while maxq and nums[maxq[-1]] <= x: maxq.pop()
        while minq and nums[minq[-1]] >= x: minq.pop()
        maxq.append(r); minq.append(r)
        while nums[maxq[0]] - nums[minq[0]] > k:
            if maxq[0] == L: maxq.popleft()
            if minq[0] == L: minq.popleft()
            t.update(P[L], -1)
            L += 1
        ans = max(ans, t.best(P[r + 1]))
        t.update(P[r + 1], 1)
    return ans

for nums, k, exp in examples:
    print(trie_window(nums, k), "kỳ vọng", exp)

7 kỳ vọng 7
6 kỳ vọng 6


**Độ phức tạp Cách 3:** mỗi thao tác trie `O(B)` với `B = 15`, tổng `O(n · B)`; bộ nhớ `O(n · B)` nút.

### 7.3. Cách 4 — Trie bền vững trên khoảng chỉ số

**Ý tưởng:** phiên bản `i` của trie chứa `P[0..i]`; mỗi nút lưu `last` = **chỉ số lớn nhất** của tiền tố đi qua nút đó. Truy vấn "max XOR với tiền tố có chỉ số trong `[L, r]`" dùng phiên bản `r`, chỉ đi vào nhánh có `last ≥ L`. Mỗi phiên bản tạo mới `B` nút (sao chép đường đi).

In [8]:
def persistent_solution(nums, k):
    n = len(nums)
    P = [0] * (n + 1)
    for i, x in enumerate(nums): P[i + 1] = P[i] ^ x
    ch = [[0, 0]]; last = [-1]           # nút 0: nút rỗng
    roots = []
    def insert(prev_root, v, idx):
        new_root = len(ch); ch.append(ch[prev_root][:]); last.append(idx)
        u, p = new_root, prev_root
        for bit in range(B - 1, -1, -1):
            b = v >> bit & 1
            nxt = len(ch); ch.append(ch[ch[p][b]][:]); last.append(idx)
            ch[u][b] = nxt
            u, p = nxt, ch[p][b]
        return new_root
    root = 0
    for i in range(n + 1):
        root = insert(root, P[i], i); roots.append(root)
    def query(rt, v, lo):
        u, res = rt, 0
        for bit in range(B - 1, -1, -1):
            b = v >> bit & 1
            o = ch[u][b ^ 1]
            if o and last[o] >= lo:
                res |= 1 << bit; u = o
            else:
                u = ch[u][b]
        return res
    Ls = left_bounds(nums, k)
    return max(query(roots[r], P[r + 1], Ls[r]) for r in range(n))

for nums, k, exp in examples:
    print(persistent_solution(nums, k), "kỳ vọng", exp)

7 kỳ vọng 7
6 kỳ vọng 6


In [9]:
class SolutionPersistent:
    def maxXor(self, nums: list[int], k: int) -> int:
        return persistent_solution(nums, k)

print([SolutionPersistent().maxXor(a, k) for a, k, _ in examples])

[7, 6]


**Độ phức tạp Cách 4:** `O(n · B)` thời gian và bộ nhớ (`(n+1)(B+1)` nút). Ưu điểm: trả lời được truy vấn khoảng **bất kỳ** (không cần cửa sổ trượt đơn điệu), ví dụ nhiều truy vấn offline `[L, r]` tùy ý.

## 8. TRACE từng bước

Trace Cách 3 trên ví dụ 2 (`nums = [5,4,5,6]`, `k = 1`), `P = [0, 5, 1, 4, 2]`.

In [10]:
def trace(nums, k):
    n = len(nums); P = [0] * (n + 1)
    for i, x in enumerate(nums): P[i + 1] = P[i] ^ x
    print("P =", P)
    t = Trie(); t.update(P[0], 1); inside = [0]
    maxq, minq = deque(), deque(); L = 0; ans = 0
    for r, x in enumerate(nums):
        while maxq and nums[maxq[-1]] <= x: maxq.pop()
        while minq and nums[minq[-1]] >= x: minq.pop()
        maxq.append(r); minq.append(r)
        while nums[maxq[0]] - nums[minq[0]] > k:
            if maxq[0] == L: maxq.popleft()
            if minq[0] == L: minq.popleft()
            t.update(P[L], -1); inside.remove(L); L += 1
        q = t.best(P[r + 1]); ans = max(ans, q)
        print(f"r={r} x={x}: cửa sổ [{L},{r}], trie chứa P{inside}={[P[i] for i in inside]}, truy vấn P[{r + 1}]={P[r + 1]} -> {q}, ans={ans}")
        t.update(P[r + 1], 1); inside.append(r + 1)
    return ans

trace([5, 4, 5, 6], 1)

P = [0, 5, 1, 4, 2]
r=0 x=5: cửa sổ [0,0], trie chứa P[0]=[0], truy vấn P[1]=5 -> 5, ans=5
r=1 x=4: cửa sổ [0,1], trie chứa P[0, 1]=[0, 5], truy vấn P[2]=1 -> 4, ans=5
r=2 x=5: cửa sổ [0,2], trie chứa P[0, 1, 2]=[0, 5, 1], truy vấn P[3]=4 -> 5, ans=5
r=3 x=6: cửa sổ [2,3], trie chứa P[2, 3]=[1, 4], truy vấn P[4]=2 -> 6, ans=6


6

Chú ý `r = 3` (`x = 6`): cửa sổ thu về `[2, 3]` (đoạn `[5, 6]`, hiệu 1) — `nums[1] = 4` bị loại vì `6 − 4 = 2 > k`. Đáp án `6` đến từ đoạn `[6]`. Với `k = 2` (ô dưới) cửa sổ giữ nguyên toàn mảng và đạt `7`.

In [11]:
trace([5, 4, 5, 6], 2)

P = [0, 5, 1, 4, 2]
r=0 x=5: cửa sổ [0,0], trie chứa P[0]=[0], truy vấn P[1]=5 -> 5, ans=5
r=1 x=4: cửa sổ [0,1], trie chứa P[0, 1]=[0, 5], truy vấn P[2]=1 -> 4, ans=5
r=2 x=5: cửa sổ [0,2], trie chứa P[0, 1, 2]=[0, 5, 1], truy vấn P[3]=4 -> 5, ans=5
r=3 x=6: cửa sổ [0,3], trie chứa P[0, 1, 2, 3]=[0, 5, 1, 4], truy vấn P[4]=2 -> 7, ans=7


7

## 9. Code nộp LeetCode

In [12]:
from collections import deque

class Solution:
    def maxXor(self, nums: list[int], k: int) -> int:
        BITS = 15
        ch0, ch1, cnt = [0], [0], [0]          # trie dạng mảng

        def update(v, d):
            u = 0
            for bit in range(BITS - 1, -1, -1):
                if v >> bit & 1:
                    if not ch1[u]:
                        ch0.append(0); ch1.append(0); cnt.append(0); ch1[u] = len(cnt) - 1
                    u = ch1[u]
                else:
                    if not ch0[u]:
                        ch0.append(0); ch1.append(0); cnt.append(0); ch0[u] = len(cnt) - 1
                    u = ch0[u]
                cnt[u] += d

        def best(v):
            u = res = 0
            for bit in range(BITS - 1, -1, -1):
                if v >> bit & 1:
                    o, s = ch0[u], ch1[u]
                else:
                    o, s = ch1[u], ch0[u]
                if o and cnt[o] > 0:
                    res |= 1 << bit; u = o
                else:
                    u = s
            return res

        n = len(nums)
        P = [0] * (n + 1)
        for i, x in enumerate(nums):
            P[i + 1] = P[i] ^ x
        update(P[0], 1)
        maxq, minq = deque(), deque()
        L = ans = 0
        for r, x in enumerate(nums):
            while maxq and nums[maxq[-1]] <= x:
                maxq.pop()
            while minq and nums[minq[-1]] >= x:
                minq.pop()
            maxq.append(r); minq.append(r)
            while nums[maxq[0]] - nums[minq[0]] > k:
                if maxq[0] == L: maxq.popleft()
                if minq[0] == L: minq.popleft()
                update(P[L], -1)
                L += 1
            ans = max(ans, best(P[r + 1]))
            update(P[r + 1], 1)
        return ans

sol = Solution()
for nums, k, exp in examples:
    got = sol.maxXor(nums, k)
    print(nums, k, "->", got, "✔" if got == exp else "✘")

[5, 4, 5, 6] 2 -> 7 ✔
[5, 4, 5, 6] 1 -> 6 ✔


## 10. Vì sao thuật toán đúng

1. **Khoảng đầu trái:** với `r` cố định, `max(nums[l..r]) − min(nums[l..r])` không giảm khi `l` giảm ⇒ tập `l` hợp lệ là `[L(r), r]` (không rỗng vì `l = r` cho hiệu 0 ≤ k). Khi `r` tăng, hiệu của `[l, r+1]` ≥ của `[l, r]` ⇒ `L(r+1) ≥ L(r)`.
2. **Deque đúng:** `maxq` (giảm dần) có đầu là max của cửa sổ hiện tại; tương tự `minq`. Khi `L` tăng, chỉ số `L` (nếu đang ở đầu) bị loại.
3. **Bất biến trie:** trước truy vấn ở bước `r`, trie chứa đúng multiset `{P[L(r)], …, P[r]}` (thêm `P[i+1]` sau bước `i`, xóa `P[L]` khi `L` tăng).
4. **Trie tham lam:** chọn bit ngược ở bit cao nhất có thể luôn tối ưu, vì bit cao lấn át mọi bit thấp hơn cộng lại.
5. Đáp án `= max_r max_{l ∈ [L(r), r]} P[r+1] ⊕ P[l]` = max trên mọi đoạn hợp lệ.

## 11. Phân tích Time / Space Complexity

Đặt `n = len(nums)`, `B = 15`.

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn (dừng sớm) | `O(n²)` | `O(1)` |
| 2. Cửa sổ + quét | `O(n · w)` | `O(n)` |
| 3. Cửa sổ + trie đếm | `O(n · B)` | `O(n · B)` |
| 4. Trie bền vững | `O(n · B)` | `O(n · B)` |

## 12. Bộ test

In [13]:
def run_all(nums, k, with_brute=True):
    res = [trie_window(nums, k), persistent_solution(nums, k), Solution().maxXor(nums, k)]
    if len(nums) <= 2000:
        res.append(window_scan(nums, k))
    if with_brute:
        res.append(brute(nums, k))
    return res

for nums, k, exp in examples:
    assert all(v == exp for v in run_all(nums, k))
print("Ví dụ đề: OK")

Ví dụ đề: OK


In [14]:
# 12.2 ca biên (tự dựng, kiểm chứng bằng vét cạn)
cases = [
    ([0], 0, 0), ([7], 0, 7),
    ([3, 3, 3], 0, 3),                    # 3⊕3 = 0, tốt nhất là một phần tử
    ([1, 2, 4, 8], 0, 8),                 # k = 0: chỉ đoạn toàn bằng nhau
    ([1, 2, 4, 8], 10**4, 15),            # mọi đoạn hợp lệ
    ([2**15 - 1, 0], 2**15 - 1, 2**15 - 1),
]
for nums, k, exp in cases:
    r = run_all(nums, k)
    assert all(v == exp for v in r), (nums, k, r)
print("Ca biên: OK")

Ca biên: OK


In [15]:
# 12.3 phản ví dụ: tham lam "chọn cửa sổ dài nhất" không cho XOR lớn nhất
def longest_window_xor(nums, k):
    Ls = left_bounds(nums, k); best_len, val = 0, 0
    for r, L in enumerate(Ls):
        if r - L + 1 > best_len:
            best_len = r - L + 1; x = 0
            for v in nums[L:r + 1]: x ^= v
            val = x
    return val

nums, k = [3, 3, 1, 1], 2
print("cửa sổ dài nhất:", longest_window_xor(nums, k), "| đúng:", Solution().maxXor(nums, k))

cửa sổ dài nhất: 0 | đúng: 3


In [16]:
# 12.4 đối chiếu ngẫu nhiên
rng = random.Random(3845)
for _ in range(400):
    n = rng.randint(1, 30)
    nums = [rng.randint(0, 31) for _ in range(n)]
    k = rng.randint(0, 31)
    res = run_all(nums, k)
    assert len(set(res)) == 1, (nums, k, res)
print("400 test nhỏ: OK")
for _ in range(20):
    nums = [rng.randint(0, 2**15 - 1) for _ in range(rng.randint(200, 800))]
    k = rng.randint(0, 2**15 - 1)
    res = run_all(nums, k)
    assert len(set(res)) == 1
print("20 test vừa: OK")

400 test nhỏ: OK


20 test vừa: OK


In [17]:
# 12.5 test lớn đúng ràng buộc
rng = random.Random(9)
nums = [rng.randint(0, 2**15 - 1) for _ in range(4 * 10**4)]
for k in (0, 1000, 2**15 - 1):
    t0 = time.time(); a = Solution().maxXor(nums, k); t1 = time.time()
    print(f"k={k}: đáp án {a}, khớp Cách 3: {a == trie_window(nums, k)}, thời gian {t1 - t0:.2f}s")

k=0: đáp án 32767, khớp Cách 3: True, thời gian 0.22s


k=1000: đáp án 32767, khớp Cách 3: True, thời gian 0.22s


k=32767: đáp án 32767, khớp Cách 3: True, thời gian 0.24s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Thêm `P[r+1]` vào trie **trước** khi truy vấn

Khi đó truy vấn có thể chọn `y = P[r+1]`, tức đoạn **rỗng** với XOR `0` — vô hại cho max (vì ≥ 0)… nhưng ngược lại, nếu quên thêm `P[0]` ban đầu thì mất các đoạn bắt đầu từ chỉ số 0.

In [18]:
def wrong_no_p0(nums, k):
    n = len(nums); P = [0] * (n + 1)
    for i, x in enumerate(nums): P[i + 1] = P[i] ^ x
    t = Trie()                                        # SAI: quên thêm P[0]
    maxq, minq = deque(), deque(); L = ans = 0
    for r, x in enumerate(nums):
        while maxq and nums[maxq[-1]] <= x: maxq.pop()
        while minq and nums[minq[-1]] >= x: minq.pop()
        maxq.append(r); minq.append(r)
        while nums[maxq[0]] - nums[minq[0]] > k:
            if maxq[0] == L: maxq.popleft()
            if minq[0] == L: minq.popleft()
            if L > 0: t.update(P[L], -1)
            L += 1
        if any(t.cnt): ans = max(ans, t.best(P[r + 1]))
        t.update(P[r + 1], 1)
    return ans

print("sai:", wrong_no_p0([6, 1], 10), "| đúng:", Solution().maxXor([6, 1], 10), "(đoạn [6, 1] bắt đầu ở chỉ số 0 cần P[0])")

sai: 1 | đúng: 7 (đoạn [6, 1] bắt đầu ở chỉ số 0 cần P[0])


### Lỗi 2 — Xóa nhầm (xóa `P[L+1]` thay vì `P[L]`) — lệch một

Cửa sổ đầu trái `L` ứng với tiền tố `P[L]` (đoạn bắt đầu từ `L`). Khi `L` tăng thì `P[L]` (giá trị **cũ**) phải bị xóa.

In [19]:
def wrong_off_by_one(nums, k):
    n = len(nums); P = [0] * (n + 1)
    for i, x in enumerate(nums): P[i + 1] = P[i] ^ x
    t = Trie(); t.update(P[0], 1)
    maxq, minq = deque(), deque(); L = ans = 0
    for r, x in enumerate(nums):
        while maxq and nums[maxq[-1]] <= x: maxq.pop()
        while minq and nums[minq[-1]] >= x: minq.pop()
        maxq.append(r); minq.append(r)
        while nums[maxq[0]] - nums[minq[0]] > k:
            if maxq[0] == L: maxq.popleft()
            if minq[0] == L: minq.popleft()
            L += 1
            t.update(P[L], -1)                        # SAI: xóa P[L] mới
        ans = max(ans, t.best(P[r + 1]))
        t.update(P[r + 1], 1)
    return ans

rng = random.Random(2)
for _ in range(3000):
    nums = [rng.randint(0, 15) for _ in range(rng.randint(1, 8))]; k = rng.randint(0, 6)
    try:
        w = wrong_off_by_one(nums, k)
    except Exception as e:
        w = f"lỗi {type(e).__name__}"
    if w != Solution().maxXor(nums, k):
        print("phản ví dụ:", nums, k, "| sai:", w, "| đúng:", Solution().maxXor(nums, k)); break

phản ví dụ: [11, 14, 8, 1, 0, 11, 14] 2 | sai: 13 | đúng: 14


### Lỗi 3 — Trie không có bộ đếm (không xóa được)

Nếu chỉ "thêm" mà không xóa, trie chứa cả tiền tố **ngoài** cửa sổ ⇒ chọn được đoạn vi phạm `max − min ≤ k`.

In [20]:
def wrong_no_delete(nums, k):
    n = len(nums); P = [0] * (n + 1)
    for i, x in enumerate(nums): P[i + 1] = P[i] ^ x
    t = Trie(); t.update(P[0], 1); ans = 0
    for r in range(n):
        ans = max(ans, t.best(P[r + 1]))              # SAI: không giới hạn cửa sổ
        t.update(P[r + 1], 1)
    return ans

print("sai:", wrong_no_delete([5, 4, 5, 6], 1), "| đúng:", Solution().maxXor([5, 4, 5, 6], 1))

sai: 7 | đúng: 6


### Lỗi 4 — Deque dùng `<` thay vì `<=` khi pop (vẫn đúng nhưng…)

Với `maxq`, pop khi `nums[maxq[-1]] < x` vẫn đúng (giữ các phần tử bằng nhau), chỉ dài hơn chút. Lỗi thật sự là **quên** kiểm tra `maxq[0] == L` trước khi `popleft` — làm mất max thật của cửa sổ.

In [21]:
def wrong_popleft(nums, k):
    maxq, minq = deque(), deque(); L = 0; res = []
    for r, x in enumerate(nums):
        while maxq and nums[maxq[-1]] <= x: maxq.pop()
        while minq and nums[minq[-1]] >= x: minq.pop()
        maxq.append(r); minq.append(r)
        while nums[maxq[0]] - nums[minq[0]] > k:
            maxq.popleft() if len(maxq) > 1 else None   # SAI: pop mà không kiểm tra chỉ số
            if minq[0] == L: minq.popleft()
            L += 1
        res.append(L)
    return res

rng = random.Random(7)
for _ in range(5000):
    a = [rng.randint(0, 9) for _ in range(rng.randint(2, 8))]; kk = rng.randint(0, 5)
    if wrong_popleft(a, kk) != left_bounds(a, kk):
        print("phản ví dụ:", a, kk, "| L(r) sai:", wrong_popleft(a, kk), "| đúng:", left_bounds(a, kk)); break

phản ví dụ: [2, 6, 0, 1] 4 | L(r) sai: [0, 0, 1, 1] | đúng: [0, 0, 2, 2]


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **XOR của đoạn** ⇒ XOR tiền tố; **max XOR** ⇒ trie nhị phân tham lam theo bit.
- **Ràng buộc `max − min ≤ k` trên đoạn** ⇒ cửa sổ trượt với hai deque đơn điệu (LC 1438).
- **Tập ứng viên thay đổi theo cửa sổ** ⇒ trie có đếm (thêm/xóa) hoặc trie bền vững (truy vấn khoảng chỉ số).
- Liên quan: LC 421 *Maximum XOR of Two Numbers*, LC 1707 *Maximum XOR With an Element From Array*.

## 15. Biến thể của bài

1. **Không có ràng buộc `k`**: max XOR đoạn con — trie trên mọi tiền tố.
2. **Ràng buộc độ dài `≤ w`** thay vì `max − min`: cửa sổ cố định, vẫn trie có đếm.
3. **Đếm số đoạn có XOR ≥ X** trong cửa sổ: trie lưu số lượng, duyệt theo bit của `X`.
4. **Nhiều truy vấn `(l, r, x)`**: "max `x ⊕ nums[i]` với `i ∈ [l, r]`" — trie bền vững.

In [22]:
# Biến thể 2: độ dài đoạn <= w
def max_xor_len_bound(nums, w):
    n = len(nums); P = [0] * (n + 1)
    for i, x in enumerate(nums): P[i + 1] = P[i] ^ x
    t = Trie(); t.update(P[0], 1); ans = 0
    for r in range(n):
        if r - w >= 0: t.update(P[r - w], -1)         # đoạn [l, r] với l >= r - w + 1
        ans = max(ans, t.best(P[r + 1])); t.update(P[r + 1], 1)
    return ans

def brute_len(nums, w):
    best = 0
    for l in range(len(nums)):
        x = 0
        for r in range(l, min(len(nums), l + w)):
            x ^= nums[r]; best = max(best, x)
    return best

rng = random.Random(15)
for _ in range(300):
    nums = [rng.randint(0, 63) for _ in range(rng.randint(1, 15))]; w = rng.randint(1, 6)
    assert max_xor_len_bound(nums, w) == brute_len(nums, w)
print("Biến thể 2 khớp vét cạn.")

Biến thể 2 khớp vét cạn.


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | Mở rộng từ mỗi `l`, dừng sớm | `O(n²)` | `O(1)` | Kiểm thử |
| 2. Cửa sổ + quét | `L(r)` + quét `P[l]` | `O(n w)` | `O(n)` | Cửa sổ hẹp |
| 3. Cửa sổ + trie đếm | Thêm/xóa tiền tố | `O(n B)` | `O(n B)` | **Lời giải chuẩn** |
| 4. Trie bền vững | Phiên bản + `last` | `O(n B)` | `O(n B)` | Truy vấn khoảng tùy ý |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output).** `maxXor([1, 2, 3], 5)` = ?

**Bài 2 (tính tay).** `maxXor([8, 1, 9, 2], 1)` — liệt kê cửa sổ `L(r)`.

**Bài 3 (điền TODO).** Viết `max_xor_pair(nums)` (LC 421) bằng trie.

**Bài 4 (tự viết lại).** Viết lại Cách 3 với cửa sổ được tính trước (`left_bounds`) rồi chỉ thao tác trie.

**Bài 5 (mini challenge).** Trả về **đoạn** (chỉ số `l, r`) đạt max XOR (bất kỳ đoạn nào nếu có nhiều).

In [23]:
# Bài 3 — TODO (không gọi hàm này khi chạy notebook)
def max_xor_pair(nums):
    # TODO: thêm từng số vào trie, truy vấn best trước khi thêm
    pass

In [24]:
# Bài 5 — TODO (không gọi hàm này khi chạy notebook)
def best_segment(nums, k):
    # TODO: trie lưu thêm chỉ số của tiền tố ở lá để truy vết l
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) `k = 5` lớn: mọi đoạn hợp lệ. (2) XOR các đoạn: `1, 2, 3, 3, 1, 0`. (3) Đáp án `3`.

**Bài 2.** (1) `8, 9` chênh 1; `1, 2` chênh 1. (2) Không có đoạn dài hơn 1 nào chứa cả 8 và 1. (3) Các đoạn hợp lệ chỉ là từng phần tử (`[8,1]`, `[1,9]`, `[9,2]` đều có hiệu > 1) ⇒ đáp án `9`.

**Bài 3.** (1) Trie rỗng ban đầu. (2) `best` trước, `update` sau. (3) LC 421 ví dụ `[3,10,5,25,2,8]` → `28` (cần 5 bit).

**Bài 4.** (1) `Ls = left_bounds(nums, k)`. (2) Giữ biến `cur_L`; khi `Ls[r] > cur_L` thì xóa dần. (3) So với `Solution`.

**Bài 5.** (1) Ở mỗi lá lưu một chỉ số còn sống (hoặc tập chỉ số). (2) Truy vấn trả về lá ⇒ `l`. (3) Kiểm tra XOR và điều kiện `max − min`.

## 19. Đáp án tham khảo cho bài tập

In [25]:
# Bài 1, 2
assert Solution().maxXor([1, 2, 3], 5) == 3 == brute([1, 2, 3], 5)
assert Solution().maxXor([8, 1, 9, 2], 1) == brute([8, 1, 9, 2], 1)
print("Bài 1:", 3, "| Bài 2:", Solution().maxXor([8, 1, 9, 2], 1))

Bài 1: 3 | Bài 2: 9


In [26]:
# Bài 3 (LC 421) — dùng trie 5 bit cho ví dụ
def max_xor_pair_sol(nums, bits=31):
    ch = [[0, 0]]; best = 0
    def ins(v):
        u = 0
        for b in range(bits - 1, -1, -1):
            c = v >> b & 1
            if not ch[u][c]: ch.append([0, 0]); ch[u][c] = len(ch) - 1
            u = ch[u][c]
    def q(v):
        u = r = 0
        for b in range(bits - 1, -1, -1):
            c = v >> b & 1
            if ch[u][c ^ 1]: r |= 1 << b; u = ch[u][c ^ 1]
            else: u = ch[u][c]
        return r
    ins(nums[0])
    for v in nums[1:]:
        best = max(best, q(v)); ins(v)
    return best

assert max_xor_pair_sol([3, 10, 5, 25, 2, 8]) == 28
print("Bài 3 OK")

Bài 3 OK


In [27]:
# Bài 4
def sol_precomputed(nums, k):
    n = len(nums); P = [0] * (n + 1)
    for i, x in enumerate(nums): P[i + 1] = P[i] ^ x
    Ls = left_bounds(nums, k); t = Trie(); t.update(P[0], 1); cur = 0; ans = 0
    for r in range(n):
        while cur < Ls[r]:
            t.update(P[cur], -1); cur += 1
        ans = max(ans, t.best(P[r + 1])); t.update(P[r + 1], 1)
    return ans

rng = random.Random(4)
for _ in range(300):
    nums = [rng.randint(0, 31) for _ in range(rng.randint(1, 25))]; k = rng.randint(0, 31)
    assert sol_precomputed(nums, k) == Solution().maxXor(nums, k)
print("Bài 4 OK")

Bài 4 OK


In [28]:
# Bài 5: trả về đoạn đạt max (dùng cửa sổ + quét để truy vết, đủ cho n vừa)
def best_segment_sol(nums, k):
    n = len(nums); P = [0] * (n + 1)
    for i, x in enumerate(nums): P[i + 1] = P[i] ^ x
    Ls = left_bounds(nums, k); best = (-1, 0, 0)
    for r in range(n):
        for l in range(Ls[r], r + 1):
            best = max(best, (P[r + 1] ^ P[l], l, r))
    return best[1], best[2], best[0]

rng = random.Random(5)
for _ in range(200):
    nums = [rng.randint(0, 31) for _ in range(rng.randint(1, 15))]; k = rng.randint(0, 31)
    l, r, v = best_segment_sol(nums, k)
    seg = nums[l:r + 1]; x = 0
    for y in seg: x ^= y
    assert x == v == Solution().maxXor(nums, k) and max(seg) - min(seg) <= k
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
P[i+1] = P[i] ^ nums[i];  trie.update(P[0], +1)
for r, x in enumerate(nums):
    cập nhật maxq (giảm dần), minq (tăng dần)
    while nums[maxq[0]] - nums[minq[0]] > k:
        bỏ L khỏi đầu deque nếu trùng; trie.update(P[L], -1); L += 1
    ans = max(ans, trie.best(P[r+1]))
    trie.update(P[r+1], +1)
```

**Checklist:**

- [ ] Thêm `P[0]` trước vòng lặp?
- [ ] Truy vấn **trước**, thêm `P[r+1]` **sau**?
- [ ] Khi `L` tăng, xóa `P[L]` (giá trị cũ)?
- [ ] Chỉ `popleft` deque khi đầu deque **bằng** `L`?
- [ ] Trie có bộ đếm, bỏ qua nhánh `cnt = 0`?